# Fase de Calibración de Hiperparámetros (Parameter Tuning)

## 1. Introducción y Justificación Metodológica
En el campo de la Computación Evolutiva, el rendimiento de un algoritmo está intrínsecamente ligado a la configuración de sus parámetros de control. Según Eiben & Smith (2015), comparar una heurística determinista (como *AMD*) contra un Algoritmo Evolutivo (EA) utilizando parámetros arbitrarios carece de rigor científico.

Este notebook tiene como objetivo ejecutar un **ajuste de parámetros (tuning)** para identificar la configuración que maximiza la robustez del algoritmo. El objetivo no es sobre-optimizar para una instancia específica, sino encontrar una configuración general capaz de explorar y explotar eficientemente el espacio de búsqueda del problema *Minimum Fill-in*.

In [1]:
import sys
import os
import itertools
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Añadimos la raíz del proyecto para importar módulos
sys.path.append('..')

from utils.graph_loader import load_pace_graph, nx_to_adj_matrix
from utils.heuristics import greedy_minimum_degree
from src.tunning import GridSearchTuner
from src.graph_chordalizer import GraphChordalizer

# Configuración visual
sns.set_theme(style="whitegrid")
print("Entorno configurado.")

Entorno configurado.


## 2. Prevención del Sobreajuste (Instancia de Sacrificio)
Para garantizar la validez de los resultados finales, se aplica el protocolo de "Instancia de Sacrificio". Se selecciona un grafo que:
1.  **No pertenece** al conjunto de prueba final (`selected_instances.csv`), evitando así el sesgo de datos (*Data Leakage*).
2.  Posee complejidad estructural moderada ($Fill\text{-}in_{AMD} > 0$).
3.  Tiene un tamaño manejable (40-150 nodos) para permitir múltiples ejecuciones en tiempo razonable.

In [2]:
DATA_DIR = "../data/raw/PACE2017B/"
SELECTED_INSTANCES_PATH = "../data/processed/selected_instances.csv"

records = []
print("Buscando un grafo 'Medio' para calibración...")

# Evitamos usar grafos seleccionados para el experimento
selected_instances = pd.read_csv(SELECTED_INSTANCES_PATH, usecols=['Filename']).values.flatten().tolist()

for f in sorted(os.listdir(DATA_DIR)):
    if f.endswith('.graph') and f not in selected_instances:
        path = os.path.join(DATA_DIR, f)
        G = load_pace_graph(path)
        G_adj = nx_to_adj_matrix(G)

        if G:
            n = G.number_of_nodes()
            e = G.number_of_edges()
            # Filtramos para no considerar grafos gigantes o diminutos
            if 40 <= n <= 150:
                _, cost = greedy_minimum_degree(G_adj, compute_cost=True)
                records.append({'Graph': f, 'Nodes': n, 'Edges':e, 'AMD_Cost': cost})

df_candidates = pd.DataFrame(records).sort_values('AMD_Cost', ascending=False).reset_index(drop=True)

print("\nCandidatos sugeridos:")
display(df_candidates)

Buscando un grafo 'Medio' para calibración...

Candidatos sugeridos:


,Graph,Nodes,Edges,AMD_Cost
0,26.graph,120,4904,254
1,89.graph,138,6338,235
2,92.graph,132,255,212
3,2.graph,129,4943,200
4,11.graph,126,1095,195
5,44.graph,111,4332,170
6,79.graph,120,4941,157
7,31.graph,100,207,151
8,8.graph,118,4880,138
9,86.graph,76,132,120


### 2.1 Selección del Grafo para Calibración de Hiperparámetros

Para realizar el ajuste fino (*tuning*) del algoritmo evolutivo, es crucial seleccionar una instancia representativa que evite sesgos en los resultados. Se analizaron los grafos disponibles en el dataset PACE 2017 y se clasificaron según su dificultad basada en el costo obtenido por la heurística constructiva (AMD).

**Criterios de Selección:**
1.  **Evitar el "Efecto Techo":** Grafos con costo AMD cercano a 0 (ej. `85.graph`, `4.graph`) son demasiado simples; cualquier configuración del GA alcanzaría el óptimo rápidamente, impidiendo distinguir qué parámetros son superiores.
2.  **Evitar Costo Computacional Excesivo:** Grafos extremadamente densos o grandes (ej. `26.graph`, costo 254) requieren tiempos de ejecución muy altos para converger, lo cual es inviable para un proceso iterativo como el *Grid Search*.
3.  **Punto de Equilibrio:** Se buscó un grafo con un costo de *fill-in* sustancial (entre 50 y 100), permitiendo margen de mejora para el algoritmo evolutivo.

**Instancia Seleccionada:** `13.graph`

| Característica | Valor | Interpretación |
| :--- |:------| :--- |
| **Nodos** | 119   | Tamaño manejable para ejecuciones iterativas. |
| **Aristas** | 161   | Densidad media. |
| **Costo Base (AMD)** | 92    | **Suficientemente alto:** Ofrece un amplio margen de mejora. Si el GA baja este costo a 80 o 60, la diferencia será estadísticamente significativa. |

Esta elección garantiza que las diferencias de rendimiento observadas en el *Grid Search* se deban a la calidad de los hiperparámetros y no a la trivialidad o imposibilidad de la instancia.

In [3]:
# Cargar grafo de "sacrificio"
GRAPH_PATH = "../data/raw/PACE2017B/13.graph"
G_tune = load_pace_graph(GRAPH_PATH)
G_tune_adj = nx_to_adj_matrix(G_tune)

# Calcular Baseline
_, amd_cost = greedy_minimum_degree(G_tune_adj, compute_cost=True)
print(f"Grafo cargado: {G_tune.number_of_nodes()} nodos.")
print(f"Meta a vencer (AMD): {amd_cost}")

Grafo cargado: 119 nodos.
Meta a vencer (AMD): 92


# 3. Diseño Experimental y Calibración de Hiperparámetros

Dado que el rendimiento de las metaheurísticas depende sensiblemente de sus hiperparámetros, hemos descartado el uso de valores estáticos por defecto (como `cx_prob=0.8` o `mut_prob=0.2`) en favor de una búsqueda adaptativa basada en el rendimiento real sobre el grafo actual.

Para lograr un equilibrio entre **tiempo de cómputo** y **calidad de la solución**, implementamos una estrategia de **Búsqueda en Malla (Grid Search) de Dos Etapas: "De Gruesa a Fina"**.

La estrategia final se divide en cuatro fases secuenciales:

1. **Fase 1: Estudio piloto:** Análisis de convergencia para límites de número de evaluaciones súmamente alto.
2. **Fase 2: Búsqueda gruesa:** Exploración rápida con pocas generaciones y saltos grandes en los parámetros para identificar regiones prometedoras.
3. **Fase 3: Búsqueda fina:** Búsqueda intensiva alrededor de la mejor configuración encontrada en la Fase 1.
4. **Fase 4: Ejecución de producción:** Ejecución final con muchas generaciones utilizando los parámetros optimizados.

A continuación, definimos la función de evaluación que servirá de base para los experimentos.

### 3.1. Fase 1: Estudio piloto

El objetivo de esta fase es identificar un presupuesto de evaluaciones en función de la aptitud (FFE, por sus siglas en inglés). La estrategia consiste en fijar un tamaño de población estándar y un límite de evaluaciones suficientemente alto para permitirnos observar cuándo las curvas de convergencia se vuelven asintóticas tras múltiples ejecuciones del algoritmo. Utilizamos los siguientes parámetros:
* Población: 100 (tamaño regular)
* Espacio de búsqueda:
    * `eval_budget` $= 100, 000$

### 3.2. Fase 2: Búsqueda Gruesa (Coarse Search)

El objetivo de esta fase es identificar si el problema responde mejor a poblaciones grandes o pequeñas, y a tasas de mutación altas o bajas.

* **Generaciones:** 30 (Bajo costo computacional).
* **Espacio de búsqueda:**
    * `pop_size` $\in [50, 200]$
    * `mut_prob` $\in [0.1, 0.4]$

In [4]:
# --- CONFIGURACIÓN FASE 1 ---
param_grid_coarse = {
    'pop_size': [50, 100, 200],     # Saltos grandes
    'mut_prob': [0.05, 0.15, 0.3] ,   # Rango amplio
    'tournsize': [2, 3, 5],
    'cx_prob': [0.8]
}

EVAL_BUDGET_COARSE = 50 * max(param_grid_coarse['pop_size'])  # mismo presupuesto para todas las configs



tuner_coarse = GridSearchTuner(
    adj_matrix=G_tune_adj,
    param_grid=param_grid_coarse,
    repetitions=3,
    max_generations=50,
    eval_budget=EVAL_BUDGET_COARSE,
)

print("--- Fase 1: Exploración Gruesa ---")
df_results_coarse = tuner_coarse.run(verbose=True)

--- Fase 1: Exploración Gruesa ---
Iniciando Tuning: 27 configs x 3 reps = 81 ejecuciones.


Running Runs:   0%|          | 0/81 [00:00<?, ?it/s]

In [6]:
# Procesamos resultados: Agrupar por configuración y sacar promedio
# Agrupamos por los parámetros variables
summary_coarse = df_results_coarse.groupby(['pop_size', 'mut_prob'])['fitness'].mean().reset_index()
summary_coarse = summary_coarse.sort_values(by='fitness', ascending=True) # Ascending=True si buscamos minimizar

# Extraemos la mejor configuración (la primera fila tras ordenar)
best_row_coarse = summary_coarse.iloc[0]
print(f"\nMejor configuración Fase 1 (Promedio): Pop={best_row_coarse['pop_size']}, Mut={best_row_coarse['mut_prob']:.2f}")
print(f"Fitness Promedio: {best_row_coarse['fitness']:.4f}")

# Mostrar tabla resumen
display(summary_coarse)


Mejor configuración Fase 1 (Promedio): Pop=200.0, Mut=0.30
Fitness Promedio: 135.3333


,pop_size,mut_prob,fitness
8,200,0.30,135.333333
5,100,0.30,138.444444
7,200,0.15,144.444444
4,100,0.15,151.555556
6,200,0.05,153.333333
2,50,0.30,158.555556
1,50,0.15,166.222222
3,100,0.05,183.111111
0,50,0.05,183.222222


### 3.3. Fase 3: Búsqueda Fina (Fine Tuning)

Basándonos en el "ganador" de la fase anterior, definimos un nuevo espacio de búsqueda más acotado alrededor de esos valores. Aumentamos el número de generaciones para asegurar que la configuración es robusta a mediano plazo.

* **Generaciones:** 80 (Costo medio).
* **Estrategia:** Se prueban valores ligeramente inferiores, iguales y superiores ($\pm \Delta$) a los ganadores de la Fase 1.

In [ ]:
# Recuperamos centros de la fase anterior
center_pop = int(best_row_coarse['pop_size'])
center_mut = float(best_row_coarse['mut_prob'])

# Creamos Grid Fina Dinámica (+- delta)
# Probamos un valor menor, el mismo y uno mayor
delta_pop = 20
delta_mut = 0.05
param_grid_fine = {
    'pop_size': [center_pop - delta_pop, center_pop, center_pop + delta_pop],
    'mut_prob': [round(center_mut - delta_mut, 2), center_mut, round(center_mut + delta_mut, 2)]
}

# Filtros de seguridad (evitar poblaciones < 10 o probabilidades negativas)
param_grid_fine['pop_size'] = sorted(list(set([p for p in param_grid_fine['pop_size'] if p >= 10])))
param_grid_fine['mut_prob'] = sorted(list(set([m for m in param_grid_fine['mut_prob'] if 0.01 <= m <= 1.0])))

print(f"--- Fase 2: Refinamiento alrededor de Pop:{center_pop}, Mut:{center_mut} ---")

# Instanciamos de nuevo la clase, ahora con configuración "robusta"
tuner_fine = GridSearchTuner(
    graph=G_tune,
    param_grid=param_grid_fine,
    repetitions=10,   # 10 para validación estadística robusta
    generations=80    # 80 para convergencia media
)

# Ejecutamos
df_results_fine = tuner_fine.run(verbose=True)

In [ ]:
# Procesamos resultados finales
summary_fine = df_results_fine.groupby(['pop_size', 'mut_prob'])['fitness'].mean().reset_index()
summary_fine = summary_fine.sort_values(by='fitness', ascending=True)

best_row_final = summary_fine.iloc[0]

print(f"\n>>> CONFIGURACIÓN ÓPTIMA V1: Pop={best_row_final['pop_size']}, Mut={best_row_final['mut_prob']:.2f}")
print(f"Fitness Promedio: {best_row_final['fitness']:.4f}")

### 3.4. Fase 4: Primera Ejecución de Producción y Análisis de Convergencia

Con los parámetros obtenidos en la Fase 2, se procedió a una ejecución extendida de 500 generaciones para evaluar el comportamiento del algoritmo a largo plazo.

El objetivo de esta etapa no es solo obtener una solución, sino **diagnosticar la capacidad del algoritmo para escapar de óptimos locales**. Para ello, se analiza la curva de convergencia (Fitness vs. Generaciones).

In [ ]:
# Instancia final directa del algoritmo
ea_diag = GraphChordalizer(G_tune)

print("--- Ejecutando Modelo Óptimo V1 ---")
best_ind_diag, log_diag = ea_diag.run_ea(
    num_generations=500,  # Convergencia profunda
    population_size=int(best_row_final['pop_size']),
    cx_prob=0.8,          # Mantenemos el default seguro
    mut_prob=float(best_row_final['mut_prob']),
    verbose=False
)

print(f"Mejor Fitness alcanzado: {best_ind_diag.fitness.values[0]}")

In [ ]:
# --- Función de Graficado de Convergencia ---
def plot_convergence(logbook):
    # Extraer datos del logbook (asumiendo estructura DEAP estándar)
    gen = logbook.select("gen")
    min_fit = logbook.select("min") # El mejor de cada generación
    avg_fit = logbook.select("avg") # El promedio de la población

    plt.figure(figsize=(10, 6))
    plt.plot(gen, min_fit, 'b-', label='Mejor Fitness (Min)')
    plt.plot(gen, avg_fit, 'r--', alpha=0.5, label='Fitness Promedio')

    plt.title('Curva de Convergencia del Algoritmo Genético')
    plt.xlabel('Generaciones')
    plt.ylabel('Fitness (Fill-in)')
    plt.legend()
    plt.grid(True, which='both', linestyle='--', linewidth=0.5)
    plt.show()

# Generamos la gráfica
plot_convergence(log_diag)

### 3.4. Diagnóstico de Estancamiento y Re-calibración Extendida

**Análisis de Resultados:**
Al observar la curva de convergencia de la sección 3.3, se detecta un comportamiento de estancamiento prematuro. Aunque el algoritmo logra reducir el *fitness* inicialmente, la curva se aplana significativamente alrededor de la generación 100, alcanzando un valor asintótico de 102.

Esto sugiere que la configuración actual de hiperparámetros, aunque estable, carece de la **presión de selección** o de la **diversidad** suficiente para escapar de este óptimo local profundo.

**Decisión de Diseño:**
Dado este hallazgo, se plantea una **Segunda Etapa de Refinamiento (Fase 4)**. A diferencia de la calibración anterior, esta nueva búsqueda integrará variables que no se consideraron inicialmente para romper el estancamiento:

1.  **Probabilidad de Cruce (`cx_prob`):** Se libera este parámetro (antes fijo en 0.8) para evaluar si un cruce más agresivo o conservador favorece la recombinación de sub-estructuras óptimas.
2.  **Mayor Carga Computacional:** Se aumenta el límite de generaciones en la búsqueda de 80 a 200 para dar tiempo a que emerjan fenotipos complejos.
3.  **Población Extendida:** Se explora incrementar el tamaño de la población para inyectar mayor material genético inicial.

In [ ]:
base_pop = int(best_row_final['pop_size'])
base_mut = float(best_row_final['mut_prob'])

param_grid_aggressive = {
    # Aumentamos drásticamente la población.
    # Si base_pop era 100, probaremos 200 y 400.
    'pop_size': [base_pop * 2, base_pop * 4],

    # Probamos mutaciones más altas para evitar que la línea roja toque la azul
    'mut_prob': [0.3, 0.5],

    # Probamos cruces variados
    'cx_prob': [0.7, 0.9]
}

print(f"--- Fase 4: Estrategia de Alta Diversidad ---")
print(f"Justificación: El modelo base convergió en 1 min. Se invierte tiempo en tamaño poblacional.")
print(f"Grid a probar: {param_grid_aggressive}")

# Instanciamos el Tuner
tuner_aggressive = GridSearchTuner(
    graph=G_tune,
    param_grid=param_grid_aggressive,
    repetitions=5,     # 5 repeticiones son suficientes si la población es gigante.
    generations=300    # 300 son suficientes, anteriormente mostramos que se estanca en 100.
)

# Ejecutar
df_results_agg = tuner_aggressive.run(verbose=True)

In [ ]:
# Procesar resultados
summary_agg = df_results_agg.groupby(['pop_size', 'mut_prob', 'cx_prob'])['fitness'].mean().reset_index()
summary_agg = summary_agg.sort_values(by='fitness', ascending=True)

best_row_agg = summary_agg.iloc[0]

print(f"\n>>> CONFIGURACIÓN 'HEAVY DUTY' GANADORA:")
print(f"Pop: {best_row_agg['pop_size']}")
print(f"Mut: {best_row_agg['mut_prob']:.2f}")
print(f"Cross: {best_row_agg['cx_prob']:.2f}")
print(f"Fitness Promedio: {best_row_agg['fitness']:.4f}")

In [ ]:
# Instancia final directa del algoritmo
ea_diag_v2 = GraphChordalizer(G_tune)

print("--- Ejecutando Modelo Óptimo V2 ---")
best_ind_diag_v2, log_diag_v2 = ea_diag.run_ea(
    num_generations=300,  # Convergencia profunda
    population_size=int(best_row_agg['pop_size']),
    cx_prob=0.9,          # Mantenemos el default seguro
    mut_prob=float(best_row_agg['mut_prob']),
    verbose=False
)

In [ ]:

print(f"Mejor Fitness alcanzado: {best_row_agg['fitness']:.4f}")
plot_convergence(log_diag_v2)

## 4. Análisis de Resultados
La métrica de desempeño seleccionada es el **MBF (Mean Best Fitness)**, definido como el promedio del mejor costo (*fill-in*) encontrado al final de las 50 generaciones.

$$MBF = \frac{1}{n} \sum_{i=1}^{n} BestFitness_i$$

Se seleccionará la configuración que minimice el MBF. En caso de resultados similares, se aplicará el principio de parsimonia, prefiriendo la configuración con menor costo computacional (ej. menor población).

In [ ]:
# Suponiendo que df_results es lo que retornó tuner.run()

# 1. Crear tabla resumen (Agregación bajo demanda)
summary = df_results.groupby(['pop_size', 'cx_prob', 'mut_prob'])['fitness'].agg(
    ['mean', 'std', 'min', 'max']
).reset_index()

summary = summary.sort_values(by='mean')
print("🏆 Top 5 Configuraciones (Promedio):")
display(summary.head(5))

# 2. Visualización Profesional (Boxplot Real)
import seaborn as sns
import matplotlib.pyplot as plt

# Creamos etiqueta legible para el eje X
df_results['Config_Label'] = df_results.apply(
    lambda x: f"P{int(x.pop_size)}_C{x.cx_prob}_M{x.mut_prob}", axis=1
)

plt.figure(figsize=(14, 6))

# Boxplot usa los datos CRUDOS para calcular cuartiles reales
sns.boxplot(x='Config_Label', y='fitness', data=df_results, palette="Set3")

# Swarmplot superpone los puntos reales (útil si N es pequeño, <20 reps)
sns.stripplot(x='Config_Label', y='fitness', data=df_results, color=".25", size=4, alpha=0.6)

plt.xticks(rotation=45, ha='right')
plt.title(f"Dispersión Real de Resultados ({len(df_results)} corridas totales)")
plt.ylabel("Fill-in Cost")
plt.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()